# Lesson 04 — Hybrid search and Reciprocal Rank Fusion

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/04-hybrid-search-rrf/colab.ipynb)

*Dense and lexical retrieval fail in opposite directions. Does combining them help? Measure it.*

**What you will learn**

- How BM25 scores documents — IDF, term-frequency saturation, length normalisation — implemented from the formula
- Why dense and BM25 scores cannot be added together
- Reciprocal Rank Fusion: the formula, the role of `k`, and an implementation by hand
- A negative result: hybrid search *lost* to plain dense retrieval here — and why
- The clue that hybrid builds a better *candidate pool*, which lesson 05 cashes in

| | |
|---|---|
| ⏱️ Time | 60 min |
| 🖥️ Runtime | CPU is fine |
| 🤖 Needs the generator (Ollama) | no — retrieval only |

**Before this:** [03-evaluation-harness](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/03-evaluation-harness/colab.ipynb)

> **How to use this notebook.** Run the cells top to bottom (`Shift+Enter`). Every
> time you see **🔮 Predict**, stop and write down what you expect *before* running
> the next cell — the gap between prediction and result is where the learning is.
> Answers to the check-your-understanding questions are hidden behind ▶ toggles.

## 0. Setup

This cell clones the course repo into the Colab runtime, installs the
dependencies, and (if `USE_LLM` is ticked) installs and starts
[Ollama](https://ollama.com) with the small `qwen2.5:1.5b` generator. It is safe
to re-run. Everything in the repo's `ragkit/` library becomes importable.

If an import fails after the install, use **Runtime → Restart session** and run
this cell again.

In [ ]:
#@title ⚙️ Setup — run me first
USE_LLM = False  #@param {type:"boolean"}
BRANCH = "main"  #@param {type:"string"}

import subprocess
import sys
from pathlib import Path


def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "ragkit").is_dir() and (p / "colab" / "colab_setup.py").exists():
            return p
    return None


REPO = _find_repo()
if REPO is None:
    REPO = Path("/content/rag-from-scratch-to-production")
    if not REPO.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH,
             "https://github.com/safar-byte/rag-from-scratch-to-production.git", str(REPO)],
            check=True,
        )
sys.path.insert(0, str(REPO / "colab"))
from colab_setup import compare_runs, eval_frames, results_frame, setup

LLM_READY = setup(llm=USE_LLM)

## 1. Two retrievers, opposite failures

| | Dense (embeddings) | Lexical (BM25) |
|---|---|---|
| Matches | **meaning** | **terms** |
| Wins on | "charged twice" → "duplicate transactions" (vocabulary mismatch) | `TX-4491`, `FUSION_CONSTANT`, version numbers, rare names |
| Fails on | identifiers, codes — tokens with no meaning | synonyms, paraphrase — "car" vs "automobile" |

Run both, merge the results, get the strengths of each. That is the theory. This lesson
is mostly about what happened when it was measured.

In [ ]:
from ragkit.pipeline import RagPipeline

pipeline = RagPipeline(strategy="dense")
if pipeline.count() == 0:
    print(pipeline.ingest())          # builds BOTH the vector index and the BM25 index
if not pipeline._bm25.count():
    pipeline._bm25.load()                 # the BM25 index persists on disk next to Chroma
chunks = pipeline._bm25._chunks           # the exact chunk list both indexes were built from
print(len(chunks), "chunks in the BM25 index")

## 2. BM25, from the formula

BM25 scores a document *d* for a query *q* by summing over the query's terms *t*:

$$\text{BM25}(d, q) = \sum_{t \in q} \text{IDF}(t) \cdot
\frac{f(t,d)\,(k_1 + 1)}{f(t,d) + k_1 \left(1 - b + b\,\frac{|d|}{\text{avgdl}}\right)}$$

Three ideas, each fixing a flaw of plain word counting:

1. **IDF** — $\log\left(1 + \frac{N - n_t + 0.5}{n_t + 0.5}\right)$ where $n_t$ is how
   many documents contain *t*. Rare terms (`TX-4491`) count a lot; common ones ("the",
   "retrieval" in a corpus about retrieval) count little.
2. **Saturation** ($k_1 \approx 1.2–1.5$) — the 10th occurrence of a word adds much less
   than the 1st. Plain counts reward keyword stuffing.
3. **Length normalisation** ($b \approx 0.75$) — a long document mentions everything
   once; divide that advantage away.

It has no idea what anything *means*, which is exactly why it is worth having.

In [ ]:
import math
import re
from collections import Counter

import numpy as np


def tokenize(text):
    return re.findall(r"[a-z0-9][a-z0-9_\-]*", text.lower())


docs_tokens = [tokenize(c.embed_text) for c in chunks]
N = len(docs_tokens)
avgdl = sum(map(len, docs_tokens)) / N
doc_freq = Counter(t for toks in docs_tokens for t in set(toks))
tfs = [Counter(toks) for toks in docs_tokens]


def idf(term):
    n = doc_freq.get(term, 0)
    return math.log(1 + (N - n + 0.5) / (n + 0.5))


def bm25_scores(query, k1=1.5, b=0.75):
    scores = np.zeros(N)
    for term in tokenize(query):
        for i, tf in enumerate(tfs):
            f = tf.get(term, 0)
            if f:
                norm = k1 * (1 - b + b * len(docs_tokens[i]) / avgdl)
                scores[i] += idf(term) * f * (k1 + 1) / (f + norm)
    return scores


for term in ["the", "retrieval", "rerank", "tx-4491", "fusion_constant"]:
    print(f"IDF({term!r:18}) = {idf(term):.2f}   (appears in {doc_freq.get(term, 0)} chunks)")

In [ ]:
def top(scores, k=5):
    order = np.argsort(-scores)[:k]
    return [(round(float(scores[i]), 2), chunks[i].doc_id) for i in order if scores[i] > 0]


for q in ["What is error code TX-4491?", "Why do I get charged twice?"]:
    print(f"{q}\n   my BM25 → {top(bm25_scores(q))}")

Compare with `ragkit`'s `Bm25Index` (built on the `bm25s` library). It adds two things
our toy lacks: **stopword removal** and **stemming**.

In [ ]:
import Stemmer

stemmer = Stemmer.Stemmer("english")
print([stemmer.stemWord(w) for w in ["chunking", "chunks", "chunked", "retrieval", "retrieving"]])

for q in ["What is error code TX-4491?", "Why do I get charged twice?"]:
    hits = pipeline._bm25.search(q, top_k=5)
    print(f"{q}\n   ragkit BM25 → {[(round(h.score, 2), h.chunk.doc_id) for h in hits]}")

Without stemming, "chunking" and "chunks" are unrelated terms and the lexical retriever
quietly stops matching the obvious things — look back at the IDF cell: our toy found
`rerank` in **0** chunks, because the corpus only ever says "reranking" and "reranker".
A query word that is not spelled exactly as in the document scores nothing.

Notice what BM25 does with "Why do I get charged twice?": it matches whatever happens
to share the words *charged* or *twice*. It has no way to know that the answer is about
**prompt caching** and paying for the same tokens repeatedly. Remember this — it is the
mechanism behind this lesson's negative result.

## 3. Why you cannot just add the scores

In [ ]:
import matplotlib.pyplot as plt

from ragkit.retrieve import Bm25Retriever, DenseRetriever

dense = DenseRetriever(pipeline._store, pipeline.embedder)
lexical = Bm25Retriever(pipeline._bm25)

q = "How does reciprocal rank fusion combine retrievers?"
d_scores = [s.score for s in dense.retrieve(q, top_k=50)]
b_scores = [s.score for s in lexical.retrieve(q, top_k=50)]

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].hist(d_scores, bins=20, color="#4C72B0"); axes[0].set_title("dense cosine (bounded, ~0.4–0.9)")
axes[1].hist(b_scores, bins=20, color="#DD8452"); axes[1].set_title("BM25 (unbounded, corpus-dependent)")
plt.show()
print(f"dense range {min(d_scores):.2f}–{max(d_scores):.2f} | BM25 range {min(b_scores):.2f}–{max(b_scores):.2f}")

A cosine of 0.82 and a BM25 score of 14.3 live on different scales. Normalising
(min-max, z-score) is fragile because BM25 has no upper bound — the maximum depends on
the corpus, the query length and the term statistics, so it shifts query to query.

## 4. Reciprocal Rank Fusion

RRF throws the scores away and uses only **ranks**:

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k + \text{rank}_r(d)}$$

with $k = 60$ by convention. A document absent from a list gets nothing from it.

The constant **damps the top ranks**: at k=60, rank 1 is worth 1/61 = 0.0164 and rank 2
is worth 1/62 = 0.0161 — nearly the same. So a document ranked moderately well by
*both* retrievers beats one ranked first by only one. Agreement between two independent
methods is strong evidence; RRF counts **votes** rather than averaging opinions. No
training, no normalisation, no tuning.

In [ ]:
def my_rrf(rankings: dict[str, list[str]], k: int = 60) -> list[tuple[str, float]]:
    fused = Counter()
    for ranking in rankings.values():
        for rank, doc in enumerate(ranking, start=1):
            fused[doc] += 1 / (k + rank)
    return fused.most_common()


toy = {"dense": ["A", "B", "C", "D"],
       "bm25":  ["E", "C", "B", "F"]}
for k in (1, 10, 60):
    print(f"k={k:3d} →", [(d, round(s, 4)) for d, s in my_rrf(toy, k)])

🔍 With k=60, **B** and **C** (ranked by both) beat **A** and **E** (each ranked first by
only one retriever). With k=1 the top ranks dominate and A wins outright. `k` decides
how much a single retriever's #1 is worth versus agreement.

The library version (`ragkit.retrieve.reciprocal_rank_fusion`) does the same but keeps
*provenance*: every fused result carries `dense_rank`, `bm25_rank` and each retriever's
RRF contribution — because "which retriever put this here?" is the question you ask
when debugging, and it is unrecoverable once the lists merge.

In [ ]:
from ragkit.retrieve import HybridRetriever

hybrid = HybridRetriever([dense, lexical], candidates=25)
results_frame(hybrid.retrieve("What does exit status 75 mean?", top_k=5), chars=60)

Read the columns: `dense_rank` and `bm25_rank` show where each retriever put the chunk
(missing = not in its top 25), `rrf_dense` / `rrf_bm25` are the two votes, and `rrf` is
their sum.

### Side by side
The lesson script shows all three rankings for a query. Try both — an identifier query
and a vocabulary-mismatch query:

In [ ]:
!{sys.executable} lessons/04-hybrid-search-rrf/run.py "What does exit status 75 mean?"

In [ ]:
!{sys.executable} lessons/04-hybrid-search-rrf/run.py "Why do I get charged twice?"

## 5. Measure it

🔮 **Predict** before running: rank `dense`, `bm25` and `hybrid` by overall recall@3,
and separately on the `vocab_mismatch` questions.

In [ ]:
from ragkit.eval.run import run_eval

runs = {}
for strategy in ["dense", "bm25", "hybrid"]:
    pipeline.set_strategy(strategy)          # reuses the loaded embedder
    runs[strategy] = run_eval(pipeline=pipeline, retrieval_only=True, quiet=True)

print("ALL QUESTIONS")
display(compare_runs(runs).round(3))
print("VOCAB_MISMATCH ONLY")
display(compare_runs(runs, kind="vocab_mismatch").round(3))
print("MULTI_HOP ONLY")
display(compare_runs(runs, kind="multi_hop").round(3))

In [ ]:
ax = compare_runs(runs)[["recall@1", "recall@3", "recall@5", "vocab R@5"]].T.plot.bar(
    figsize=(10, 3.5), rot=0)
ax.set_ylim(0.5, 1.02)
ax.set_title("dense vs bm25 vs hybrid")
plt.show()

### What the repo measured

| Strategy | R@1 | R@3 | R@5 | nDCG@5 | vocab R@3 |
|---|---|---|---|---|---|
| dense | **0.824** | **0.932** | 0.932 | **0.925** | **0.750** |
| bm25 | 0.689 | 0.919 | 0.919 | 0.859 | 0.625 |
| hybrid | **0.824** | 0.905 | **0.946** | **0.925** | 0.625 |

(Measured on an earlier, 37-question version of the golden set; your exact numbers
over 44 questions will differ — compare the *pattern*.)

**Hybrid did not beat dense at the top of the ranking.** It tied at R@1 and was *worse*
at R@3 and on vocabulary-mismatch R@3 — the very category hybrid search is supposed to
help with. If this repo were selling hybrid search, that row would not be there. It is
there because a negative result is a finding.

Check the same columns in your run above. When this notebook was written (44
questions): R@1 tied, R@3 **worse** (0.920 vs 0.943), vocab R@3 **worse** (0.625 vs
0.750) — and R@5 *better* (0.977 vs 0.943). Hold on to that last number; section 7
explains it.

## 6. Why it went wrong

**RRF weights every retriever equally.** That is its virtue when both are roughly as
good, and its failure mode when one is systematically bad for a query type. On
vocabulary-mismatch questions BM25 is not merely unhelpful, it is **confidently wrong** —
it ranks documents that share surface words with the query and nothing else. Look:

In [ ]:
import pandas as pd

from ragkit.eval.golden import load_golden

rows = []
for q in [q for q in load_golden() if q.kind == "vocab_mismatch"]:
    rank_in = {}
    for name, retriever in [("dense", dense), ("bm25", lexical), ("hybrid", hybrid)]:
        docs = list(dict.fromkeys(s.chunk.doc_id for s in retriever.retrieve(q.question, top_k=20)))
        rank_in[name] = next((i for i, d in enumerate(docs, 1) if d in q.relevant), None)
    rows.append({"question": q.question[:60], "answer doc": q.relevant[0], **rank_in})
pd.DataFrame(rows)

Wherever BM25 puts the right document deep (or nowhere), equal-weight fusion drags the
good dense ranking down with it.

Two fixes, and only one is honest:

- **Weighted RRF** — give dense more votes. Tempting, and it is tuning a constant
  against 44 questions, which is how you overfit a benchmark (try it in the exercise).
- **Routing** — send identifier-shaped queries to BM25 and prose questions to dense.
  Principled, decided per query, and it is lesson 09.

## 7. The result that changes the conclusion

Look again at the overall table: hybrid's recall@5 can beat dense's even when its
recall@3 is worse. *Worse near the top, better deeper* is a clue: hybrid is not
producing a better **ranking**, it is producing a better **pool** of candidates. Check
that directly — recall at depth 10 and 20:

In [ ]:
from ragkit.eval.metrics import recall_at_k

answerable = [q for q in load_golden() if q.relevant]
pool = {}
for name, retriever in [("dense", dense), ("bm25", lexical), ("hybrid", hybrid)]:
    row = {}
    for depth in (3, 5, 10, 20):
        row[f"recall@{depth}"] = np.mean([
            recall_at_k(list(dict.fromkeys(s.chunk.doc_id for s in retriever.retrieve(q.question, top_k=depth))),
                        q.relevant, depth)
            for q in answerable])
    pool[name] = row
pd.DataFrame(pool).T.round(3)

If hybrid's recall at depth 10–20 is at least as good as dense's, it has assembled a
pool that *contains* more of the right documents — it just orders them worse. A
stronger ranker on top of that pool could have the best of both. That is exactly what
lesson 05 tests, and it is what rescues hybrid search here.

## 8. Implementation notes worth stealing

- **Both indexes are built from one chunk list** (`RagPipeline.ingest`). If they
  disagreed about chunk boundaries, fusion would merge rankings over two different
  corpora — and the symptom would be an unexplained quality regression, not an error.
- **BM25 indexes `embed_text`, not `text`**, so lesson 07's context prefixes help the
  lexical index too.
- **Stemming is on.**
- **`candidates`** (how deep each retriever goes before fusion) is the real recall
  ceiling: RRF can only rank documents that appeared in at least one list.

## 9. Check your understanding

**1. Why does RRF use 1/(k + rank) rather than the raw scores?**

<details><summary>▶ Show answer</summary>

Raw scores from different retrievers are on incomparable scales (cosine is bounded,
BM25 is unbounded and corpus-dependent), and normalising them is fragile. Ranks are
always comparable. The constant k damps the top ranks so agreement between
retrievers outweighs a single retriever's top pick.

</details>

**2. Hybrid lost on vocabulary-mismatch questions. Explain the mechanism in one sentence.**

<details><summary>▶ Show answer</summary>

BM25 confidently ranks documents that share surface words but not meaning, and
equal-weight RRF lets that bad ranking pull the correct dense results down.

</details>

**3. Why is "tune the RRF weights until hybrid wins" a bad idea here?**

<details><summary>▶ Show answer</summary>

With 44 questions — most of them saturated — you would be fitting a constant to a
handful of questions. It would look like a gain on this benchmark and generalise to
nothing. It also applies one global compromise to every query, when the evidence
says the right choice depends on the *shape* of the query (routing, lesson 09).

</details>

## ✍️ Exercises

1. **Find a query where each retriever wins.** Identifiers are the easy case for BM25 —
   find a harder one.
2. **Tune k.** Re-run the evaluation with `HybridRetriever(..., k=10)` and `k=100`.
   Did anything change by more than one question? What does that tell you?
3. **Weighted fusion.** Complete the cell below (dense counts `w` times as much as
   BM25). Find a weight that "wins" — then explain why you shouldn't ship it.

In [ ]:
from ragkit.types import Scored


class WeightedHybrid:
    name = "weighted-hybrid"

    def __init__(self, dense_weight=2.0, k=60, candidates=25):
        self.w, self.k, self.n = dense_weight, k, candidates

    def retrieve(self, query, top_k=5, where=None):
        fused = {}
        for weight, retriever in [(self.w, dense), (1.0, lexical)]:
            for rank, s in enumerate(retriever.retrieve(query, top_k=self.n), 1):
                entry = fused.setdefault(s.chunk.chunk_id, Scored(chunk=s.chunk, score=0.0))
                entry.score += weight / (self.k + rank)
        ordered = sorted(fused.values(), key=lambda s: s.score, reverse=True)[:top_k]
        for rank, s in enumerate(ordered, 1):
            s.rank = rank
        return ordered


weighted_runs = {}
for w in (1.0, 2.0, 4.0):
    pipeline._retriever = WeightedHybrid(dense_weight=w)
    weighted_runs[f"dense weight {w}"] = run_eval(pipeline=pipeline, retrieval_only=True, quiet=True)
compare_runs(weighted_runs).round(3)

## Key takeaways

1. BM25 = IDF × saturating term frequency × length normalisation; it matches terms, not meaning.
2. Dense and BM25 scores are incomparable; RRF fuses by rank and rewards agreement.
3. Equal-weight fusion propagates a confidently-wrong retriever — hybrid *lost* to dense here.
4. Worse at the top but better deeper means a better *pool*, not a better ranking — lesson 05 exploits that.
5. Prefer a principled per-query decision (routing) over a global constant tuned to a small benchmark.

## Next

**[Lesson 05 — Reranking](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/05-reranking/colab.ipynb)**